# Kiểm thử gauss_seidel: năm hệ phương trình

Chạy **Run All Cells** từ thư mục `Gau-Jacobi`. Kiểm tra chữ ký `(A, b, x0, eps, n_max)`, nghiệm tham chiếu NumPy, số bước lặp, sai số và các ca biên.

In [ ]:
import ast
from pathlib import Path

import ipytest
import nbformat
import numpy as np
import pytest

ipytest.autoconfig()


# Chỉ đọc định nghĩa hàm từ notebook, không chạy phần ví dụ.
def load_gauss_seidel(filename):
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")

    notebook = nbformat.read(path, as_version=4)
    namespace = {"np": np}

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        tree = ast.parse(cell.source)

        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == "gauss_seidel":
                module = ast.Module(body=[node], type_ignores=[])
                ast.fix_missing_locations(module)

                exec(compile(module, str(path), "exec"), namespace)
                return namespace["gauss_seidel"]

    raise ValueError(f"Không tìm thấy hàm gauss_seidel trong {filename}")


# Tải hai phiên bản Gauss-Seidel
gauss_seidel_v1 = load_gauss_seidel("gauss_seidel_v1.ipynb")
gauss_seidel_v2 = load_gauss_seidel("gauss_seidel_v2.ipynb")


# Năm hệ phương trình từ TranLe và NguyenBinh
HE_PHUONG_TRINH = [
    pytest.param(
        np.array([
            [10, -1, 2, 0],
            [-1, 11, -1, 3],
            [2, -1, 10, -1],
            [0, 3, -1, 8],
        ], dtype=float),
        np.array([6, 25, -11, 15], dtype=float),
        1e-5,
        np.array([1, 2, -1, 1], dtype=float),
        id="He_1_nghiem_dung",
    ),
    pytest.param(
        np.array([
            [1.02, -0.25, -0.30],
            [-0.41, 1.13, -0.15],
            [-0.25, -0.14, 1.21],
        ], dtype=float),
        np.array([0.515, 1.555, 2.780], dtype=float),
        1e-3,
        None,
        id="He_2_epsilon_1e-3",
    ),
    pytest.param(
        np.array([
            [10.9, 1.2, 2.1, 0.9],
            [1.2, 11.2, 1.5, 2.5],
            [2.1, 1.5, 9.8, 1.3],
            [0.9, 2.5, 1.3, 12.1],
        ], dtype=float),
        np.array([-7, 5.3, 10.3, 24.6], dtype=float),
        1e-3,
        None,
        id="He_3_epsilon_1e-3",
    ),
    pytest.param(
        np.array([
            [4, 0.24, -0.08, 0.16],
            [0.09, 3, -0.155, -0.12],
            [0.04, -0.08, -4, 0.06],
            [0.02, 0.06, 0.04, -10],
        ], dtype=float),
        np.array([8, 9, 20, 1], dtype=float),
        0.5e-3,
        None,
        id="He_4_epsilon_0.5e-3",
    ),
    pytest.param(
        np.array([[4, -1, 1], [-1, 4, -2], [1, -2, 4]], dtype=float),
        np.array([12, -1, 5], dtype=float),
        1e-4, None, id="He_5_NguyenBinh_epsilon_1e-4",
    ),
]


# Lưu kết quả kiểm thử
KET_QUA = []


# 5 hệ × 2 phiên bản = 10 ca kiểm thử nghiệm
@pytest.mark.parametrize(
    "solver",
    [gauss_seidel_v1, gauss_seidel_v2],
    ids=["Gauss_Seidel_v1", "Gauss_Seidel_v2"],
)
@pytest.mark.parametrize("A, b, esl, expected", HE_PHUONG_TRINH)
def test_bon_he_phuong_trinh(solver, A, b, esl, expected, capsys, request):

    n_max = 100
    x0 = np.zeros(len(b))

    if expected is None:
        expected = np.linalg.solve(A, b)

    # Gọi hàm theo chữ ký mới
    result, k, err = solver(A, b, x0, eps=esl, n_max=n_max)

    output = capsys.readouterr().out

    # Kiểm tra nghiệm
    assert result is not None
    assert np.all(np.isfinite(result))

    np.testing.assert_allclose(
        result,
        expected,
        atol=esl,
        rtol=0
    )

    # Kiểm tra số bước lặp
    so_buoc_lap = output.count("KQ lần lặp thứ")

    assert so_buoc_lap == k
    assert 0 <= k <= n_max

    # Kiểm tra sai số trả về
    assert np.isfinite(err)
    assert err >= 0

    # Kiểm tra điều kiện dừng
    assert err < esl

    # Lưu kết quả
    KET_QUA.append({
        "he": int(request.node.callspec.id.split("_")[1]),
        "phien_ban": (
            "Gauss-Seidel v1"
            if solver is gauss_seidel_v1
            else "Gauss-Seidel v2"
        ),
        "result": result.copy(),
        "k": k,
        "err": err,
    })


# 2 ca kiểm thử điều kiện q < 1
@pytest.mark.parametrize(
    "solver",
    [gauss_seidel_v1, gauss_seidel_v2],
    ids=["Gauss_Seidel_v1", "Gauss_Seidel_v2"],
)
def test_khong_thoa_dieu_kien_q(solver):

    A = np.array([
        [1, 2],
        [2, 1]
    ], dtype=float)

    b = np.array([3, 3], dtype=float)
    x0 = np.zeros(len(b))

    # Hàm mới phát sinh ValueError
    with pytest.raises(ValueError):
        solver(A, b, x0, eps=1e-6, n_max=10)


# Chạy toàn bộ ca kiểm thử

# Phần dư nhỏ chưa đảm bảo sai số nghiệm nhỏ khi hệ số của A nhỏ.
@pytest.mark.parametrize("solver", [gauss_seidel_v1, gauss_seidel_v2])
def test_phan_du_nho_nhung_nghiem_chua_dat(solver):
    A = np.diag([1e-8, 2e-8])
    x, k, err = solver(A, np.array([1e-8, 2e-8]), np.zeros(2), eps=1e-5, n_max=10)
    np.testing.assert_allclose(x, np.ones(2), atol=1e-5, rtol=0)
    assert k > 0

@pytest.mark.parametrize("solver", [gauss_seidel_v1, gauss_seidel_v2])
@pytest.mark.parametrize("b, eps", [(np.array([[1.0], [1.0]]), 1e-5), (np.array([]), 1e-5), (np.ones(2), np.nan), (np.array([np.inf, 1]), 1e-5)])
def test_du_lieu_khong_hop_le(solver, b, eps):
    with pytest.raises(ValueError):
        solver(np.eye(2), b, np.zeros(2), eps=eps, n_max=10)

@pytest.mark.parametrize("solver", [gauss_seidel_v1, gauss_seidel_v2])
def test_gioi_han_buoc_lap_va_khong_doi_x0(solver):
    A = np.array([[4., 1.], [1., 3.]])
    x0 = np.zeros(2)
    result, k, err = solver(A, np.ones(2), x0, eps=1e-15, n_max=1)
    np.testing.assert_array_equal(x0, np.zeros(2))
    assert k == 1 and err > 1e-15
    assert np.max(np.abs(result - np.linalg.solve(A, np.ones(2)))) <= err

exit_code = ipytest.run("-q")

assert exit_code == pytest.ExitCode.OK, "Có ca kiểm thử thất bại."


# Hiển thị kết quả
for item in KET_QUA:

    nghiem = "; ".join(
        f"{value:.6f}" for value in item["result"]
    )

    print(f"Hệ {item['he']} — {item['phien_ban']}")
    print(f"Nghiệm: [{nghiem}]")
    print(f"Số lần lặp: {item['k']}")
    print(f"Sai số: {item['err']:.6f}\n")
